In [0]:
SELECT
  ROUND(SUM(ticket_price - discount + tax) ,2) AS total_net_revenue,
  COUNT(DISTINCT booking_id) AS distinct_bookings,
  COUNT(DISTINCT passenger_id) AS active_passengers,
  ROUND(AVG(net_revenue),2) AS average_booking_value
FROM tpca.gold.fact_bookings

In [0]:
SELECT
  YEAR(booking_date) AS booking_year,
  MONTH(booking_date) AS booking_month,
  DATE_FORMAT(booking_date, 'yyyy-MM') AS year_month,
  COUNT(DISTINCT booking_id) AS bookings,
  ROUND(SUM(net_revenue), 2) AS monthly_net_revenue
FROM tpca.gold.fact_bookings
GROUP BY YEAR(booking_date), MONTH(booking_date), DATE_FORMAT(booking_date, 'yyyy-MM')
ORDER BY booking_year, booking_month

In [0]:
WITH annual AS (
  SELECT
    YEAR(booking_date) AS booking_year,
    ROUND(SUM(net_revenue), 2) AS annual_net_revenue
  FROM tpca.gold.fact_bookings
  GROUP BY YEAR(booking_date)
)
SELECT
  booking_year,
  annual_net_revenue,
  LAG(annual_net_revenue) OVER (ORDER BY booking_year) AS prior_year_revenue,
  ROUND(
    (annual_net_revenue - LAG(annual_net_revenue) OVER (ORDER BY booking_year))
      / LAG(annual_net_revenue) OVER (ORDER BY booking_year) * 100,
    2
  ) AS yoy_revenue_change_pct
FROM annual
ORDER BY booking_year


In [0]:
SELECT
  p.travel_purpose,
  COUNT(DISTINCT fb.booking_id) AS bookings,
  ROUND(SUM(fb.net_revenue), 2) AS total_net_revenue
FROM tpca.gold.fact_bookings fb
JOIN tpca.gold.dim_passenger p
  ON fb.passenger_id = p.passenger_id
GROUP BY p.travel_purpose
ORDER BY total_net_revenue DESC

In [0]:
WITH passenger_info AS (
SELECT passenger_id, passenger_name
FROM tpca.gold.dim_passenger
) 
SELECT p.passenger_id, p.passenger_name, round(sum(fb.net_revenue),2) as total_revenue
FROM tpca.gold.fact_bookings fb
JOIN passenger_info p
  ON fb.passenger_id = p.passenger_id
GROUP BY p.passenger_id, p.passenger_name
ORDER BY total_revenue DESC
LIMIT 10

In [0]:
WITH passenger_revenue AS (
  SELECT
    p.passenger_id,
    p.passenger_name,
    ROUND(SUM(fb.net_revenue), 2) AS passenger_revenue
  FROM tpca.gold.fact_bookings fb
  JOIN tpca.gold.dim_passenger p
    ON fb.passenger_id = p.passenger_id
  GROUP BY p.passenger_id, p.passenger_name
),
ranked AS (
  SELECT
    passenger_id,
    passenger_name,
    passenger_revenue,
    ROUND(SUM(passenger_revenue) OVER (), 2) AS total_revenue,
    DENSE_RANK() OVER (ORDER BY passenger_revenue DESC) AS revenue_rank
  FROM passenger_revenue
)
SELECT
  passenger_id,
  passenger_name,
  passenger_revenue,
  total_revenue,
  ROUND(passenger_revenue / total_revenue * 100, 2) AS revenue_share_pct
FROM ranked
WHERE revenue_rank <= 10
ORDER BY revenue_rank

In [0]:
SELECT
  r.route_id,
  r.route_name,
  COUNT(DISTINCT fb.booking_id) AS bookings,
  ROUND(SUM(fb.net_revenue), 2) AS total_net_revenue
FROM tpca.gold.fact_bookings fb
JOIN tpca.gold.dim_route r
  ON fb.route_id = r.route_id
GROUP BY r.route_id, r.route_name
ORDER BY total_net_revenue DESC
LIMIT 10


In [0]:
SELECT
  r.route_id,
  r.route_name,
  COUNT(DISTINCT fb.booking_id) AS bookings,
  ROUND(SUM(fb.net_revenue), 2) AS total_net_revenue
FROM tpca.gold.fact_bookings fb
JOIN tpca.gold.dim_route r
  ON fb.route_id = r.route_id
GROUP BY r.route_id, r.route_name
ORDER BY total_net_revenue ASC
LIMIT 10


In [0]:
SELECT r.route_type, sum(fb.net_revenue) as total_revenue
FROM tpca.gold.dim_route r
JOIN tpca.gold.fact_bookings fb
GROUP BY r.route_type
ORDER BY total_revenue DESC

In [0]:
SELECT a.airline_id, a.airline_name, sum(fb.net_revenue) as total_revenue
FROM tpca.gold.dim_airline a
JOIN tpca.gold.fact_bookings fb
  ON a.airline_id = fb.airline_id
GROUP BY a.airline_id, a.airline_name
ORDER BY total_revenue DESC
LIMIT 10

In [0]:
WITH airline_revenue AS (
  SELECT a.airline_id, a.airline_name, ROUND(SUM(fb.net_revenue), 2) AS total_revenue
  FROM tpca.gold.fact_bookings fb
  JOIN tpca.gold.dim_airline a ON fb.airline_id = a.airline_id
  GROUP BY a.airline_id, a.airline_name
),
ranked AS (
  SELECT airline_id, airline_name, total_revenue,
    ROUND(SUM(total_revenue) OVER (), 2) AS overall_revenue,
    DENSE_RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank
  FROM airline_revenue
)
SELECT airline_id, airline_name, total_revenue, overall_revenue,
  ROUND(total_revenue / overall_revenue * 100, 2) AS revenue_share_pct
FROM ranked
WHERE revenue_rank <= 10
ORDER BY revenue_rank

In [0]:
SELECT
  passenger_region,
  COUNT(DISTINCT booking_id) AS bookings,
  ROUND(SUM(net_revenue), 2) AS total_net_revenue
FROM tpca.gold.fact_bookings
GROUP BY passenger_region
ORDER BY total_net_revenue DESC

In [0]:
SELECT
  passenger_nation,
  COUNT(DISTINCT booking_id) AS bookings,
  ROUND(SUM(net_revenue), 2) AS total_net_revenue
FROM tpca.gold.fact_bookings
GROUP BY passenger_nation
ORDER BY total_net_revenue DESC

In [0]:
SELECT
  refund_flag,
  COUNT(DISTINCT booking_id) AS bookings,
  SUM(seats) AS seats_sold,
  ROUND(SUM(net_revenue), 2) AS total_net_revenue
FROM tpca.gold.fact_bookings
GROUP BY refund_flag
ORDER BY total_net_revenue DESC

In [0]:
SELECT
  p.travel_purpose,
  ROUND(AVG(fb.discount), 4) AS avg_discount
FROM tpca.gold.fact_bookings fb
JOIN tpca.gold.dim_passenger p ON fb.passenger_id = p.passenger_id
GROUP BY p.travel_purpose
ORDER BY avg_discount DESC

In [0]:
SELECT
  cabin_class,
  ROUND(AVG(DATEDIFF(arrival_date, departure_date)), 2) AS avg_flight_days
FROM tpca.gold.fact_bookings
GROUP BY cabin_class
ORDER BY avg_flight_days DESC

In [0]:
SELECT
  booking_priority,
  COUNT(DISTINCT booking_id) AS bookings,
  ROUND(SUM(net_revenue), 2) AS total_net_revenue
FROM tpca.gold.fact_bookings
GROUP BY booking_priority
ORDER BY total_net_revenue DESC

In [0]:
SELECT
  DATE_FORMAT(booking_date, 'yyyy-MM') AS year_month,
  booking_status,
  COUNT(DISTINCT booking_id) AS bookings,
  ROUND(SUM(net_revenue), 2) AS total_net_revenue
FROM tpca.gold.fact_bookings
GROUP BY DATE_FORMAT(booking_date, 'yyyy-MM'), booking_status
ORDER BY year_month, booking_status

In [0]:
WITH route_purpose AS (
  SELECT
    p.travel_purpose,
    r.route_id,
    r.route_name,
    ROUND(SUM(fb.net_revenue), 2) AS total_net_revenue
  FROM tpca.gold.fact_bookings fb
  JOIN tpca.gold.dim_passenger p ON fb.passenger_id = p.passenger_id
  JOIN tpca.gold.dim_route r ON fb.route_id = r.route_id
  GROUP BY p.travel_purpose, r.route_id, r.route_name
)
SELECT
  travel_purpose,
  route_id,
  route_name,
  total_net_revenue,
  RANK() OVER (PARTITION BY travel_purpose ORDER BY total_net_revenue DESC) AS revenue_rank
FROM route_purpose
ORDER BY travel_purpose, revenue_rank

In [0]:
SELECT
  passenger_region,
  airline_region,
  COUNT(DISTINCT booking_id) AS bookings,
  ROUND(SUM(net_revenue), 2) AS total_net_revenue
FROM tpca.gold.fact_bookings
GROUP BY passenger_region, airline_region
ORDER BY total_net_revenue DESC